In [1]:
import gymnasium as gym
import numpy as np
import os
import matplotlib.pyplot as plt
import time
from IPython.display import clear_output
import pandas as pd

In [2]:
env = gym.make("MountainCar-v0", render_mode="rgb_array", max_episode_steps=-1)
env.reset()

(array([-0.54609346,  0.        ], dtype=float32), {})

In [3]:
gamma = 0.99
epsilon = 0.9

max_epsilon = 0.9
min_epsilon = 0.0

decay = 0.003

learning_rate = 0.2

total_episodes = 15000
max_steps = 200

In [4]:
bins_position_amount = 50
bins_velocity_amount = 25

max_position = env.observation_space.high[0]
min_position = env.observation_space.low[0]
max_velocity = env.observation_space.high[1]
min_velocity = env.observation_space.low[1]

bins_position = np.linspace(min_position, max_position, bins_position_amount)
bins_velocity = np.linspace(min_velocity, max_velocity, bins_velocity_amount)

In [5]:
def calculate_value(Q, state, action, reward, next_state, alpha, gamma):
    Q[state[0], state[1], action] = Q[state[0], state[1], action] + alpha * (reward + gamma * np.max(Q[next_state[0], next_state[1]]) - Q[state[0], state[1], action])
    return Q

In [6]:
def get_state(observation):
    position = np.digitize(observation[0], bins=bins_position) - 1
    velocity = np.digitize(observation[1], bins=bins_velocity) - 1
    return position, velocity

In [7]:
q_table = np.zeros((bins_position_amount, bins_velocity_amount, env.action_space.n))
wins = []

for episode in range(total_episodes):
    obs, _ = env.reset()
    state = get_state(obs)
    won = False

    for step in range(max_steps):
        if np.random.uniform(0, 1) < epsilon:
            action = env.action_space.sample()
        else:
            action = np.argmax(q_table[state])

        obs, reward, terminated, truncated, _ = env.step(action)
        next_state = get_state(obs)

        target = reward if terminated else reward + gamma * np.max(q_table[next_state])
        q_table[state][action] += learning_rate * (target - q_table[state][action])

        state = next_state
        if terminated:
            won = True
            break

    wins.append(won)
    epsilon = min_epsilon + (max_epsilon - min_epsilon) * np.exp(-decay * episode)

    if (episode + 1) % 500 == 0:
        print(f"Episode {episode+1}: win rate (last 500) = {np.mean(wins[-500:]):.2f}, epsilon = {epsilon:.3f}")

Episode 500: win rate (last 500) = 0.00, epsilon = 0.201
Episode 1000: win rate (last 500) = 0.01, epsilon = 0.045
Episode 1500: win rate (last 500) = 0.08, epsilon = 0.010
Episode 2000: win rate (last 500) = 0.19, epsilon = 0.002
Episode 2500: win rate (last 500) = 0.35, epsilon = 0.000
Episode 3000: win rate (last 500) = 0.37, epsilon = 0.000
Episode 3500: win rate (last 500) = 0.55, epsilon = 0.000
Episode 4000: win rate (last 500) = 0.66, epsilon = 0.000
Episode 4500: win rate (last 500) = 0.91, epsilon = 0.000
Episode 5000: win rate (last 500) = 0.90, epsilon = 0.000
Episode 5500: win rate (last 500) = 0.91, epsilon = 0.000
Episode 6000: win rate (last 500) = 0.94, epsilon = 0.000
Episode 6500: win rate (last 500) = 0.92, epsilon = 0.000
Episode 7000: win rate (last 500) = 0.54, epsilon = 0.000
Episode 7500: win rate (last 500) = 0.87, epsilon = 0.000
Episode 8000: win rate (last 500) = 0.92, epsilon = 0.000
Episode 8500: win rate (last 500) = 0.84, epsilon = 0.000
Episode 9000: w

Coming back to it after a while and getting some cleanup assistance from Claude I managed to finish it up. (I just didn't notice I was resetting my Q table and was losing my mind but it worked out)

it very slightly overtrained at the end but actually wasn't too bad even without some automatic cutoff. I have to assume mostly luck.